# 03 — Curia Vista ↔ APS Mapping Evaluation

**Purpose:** Inspect the quality of the generated `parameters/*_to_aps.json` mapping files.

**Inputs:**
- `parameters/curia_vista_to_aps.json` (28 CHE labels)
- `parameters/cantonal_sg_to_aps.json` (10 SG labels)
- `parameters/cantonal_tg_to_aps.json` (118 TG labels)
- `parameters/cantonal_zh_to_aps.json` (81 ZH labels)
- `parameters/mapping_metadata.json` (coverage stats)
- `parameters/validation_report.json` (federal + PolitMonitor checks)

**Expected output:** summary tables and figures; no file writes.

**How to run:** `jupyter notebook notebooks/03_mapping_eval.ipynb` from the project root.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PARAMS = PROJECT_ROOT / 'parameters'
print('Project root:', PROJECT_ROOT)

## 1. Load all mapping files and metadata

In [ ]:
mappings = {}
for p in sorted(PARAMS.glob('*_to_aps.json')):
    data = json.loads(p.read_text(encoding='utf-8'))
    scheme = data['metadata']['source_scheme']
    mappings[scheme] = data
    print(f"{p.name:35s}  scheme={scheme:30s}  n={len(data['mappings'])}")

metadata = json.loads((PARAMS / 'mapping_metadata.json').read_text(encoding='utf-8'))
print('\nGlobal metadata:')
print(json.dumps(metadata['coverage'], indent=2))

## 2. Per-scheme stats

In [ ]:
rows = []
for scheme, data in mappings.items():
    for m in data['mappings']:
        rows.append({
            'scheme': scheme,
            'source_label': m['source_label_de'],
            'n_primary': len(m['aps_primary']),
            'n_secondary': len(m['aps_secondary']),
            'has_error': bool(m.get('error')),
        })
df = pd.DataFrame(rows)
summary = df.groupby('scheme').agg(
    n_labels=('source_label', 'count'),
    mean_primary=('n_primary', 'mean'),
    mean_secondary=('n_secondary', 'mean'),
    n_errors=('has_error', 'sum'),
).round(2)
summary

## 3. Distribution of primaries + secondaries per label

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['n_primary'].hist(ax=axes[0], bins=range(0, df['n_primary'].max()+2), edgecolor='black')
axes[0].set_title('Primaries per source label')
axes[0].set_xlabel('# primary APS codes')
df['n_secondary'].hist(ax=axes[1], bins=range(0, df['n_secondary'].max()+2), edgecolor='black')
axes[1].set_title('Secondaries per source label')
axes[1].set_xlabel('# secondary APS codes')
plt.tight_layout()
plt.show()

## 4. Curia Vista — full table (28 labels)

In [ ]:
che = mappings['curia_vista_subject_areas']['mappings']
che_rows = [
    {
        'label': m['source_label_de'],
        'affairs': m['source_affair_count'],
        'primary': ', '.join(m['aps_primary']),
        'n_secondary': len(m['aps_secondary']),
    }
    for m in che
]
pd.DataFrame(che_rows).sort_values('affairs', ascending=False)

## 5. Validation results (from `validate.py`)

In [ ]:
vrep = json.loads((PARAMS / 'validation_report.json').read_text(encoding='utf-8'))
print('Federal affairs coverage:', vrep['federal_coverage_pct'], '%')
print('PolitMonitor digi share:', vrep['politmonitor_digitalisierung']['digi_related_share_pct'], '%')
print('\nTop APS codes on digitalisation-tagged federal affairs:')
for e in vrep['politmonitor_digitalisierung']['top_aps_codes'][:10]:
    marker = '*' if e['is_digi'] else ' '
    print(f"  {marker} {e['code']}: {e['count']}")

## 6. APS codes with most source-label coverage (fan-in)

In [ ]:
from collections import Counter
primary_fan_in = Counter()
secondary_fan_in = Counter()
for data in mappings.values():
    for m in data['mappings']:
        primary_fan_in.update(m['aps_primary'])
        secondary_fan_in.update(m['aps_secondary'])

print('Top 10 APS codes by primary fan-in (how many source labels map to them):')
for code, n in primary_fan_in.most_common(10):
    print(f'  {code}: {n} source labels')

print('\nTop 10 APS codes by secondary fan-in:')
for code, n in secondary_fan_in.most_common(10):
    print(f'  {code}: {n} source labels')

## Summary

Run all cells to get a snapshot of mapping quality. Key metrics:

- **Total source labels mapped:** sum across all schemes (target: 237)
- **APS coverage:** fraction of 165 APS topics reachable (target: ≥95%)
- **Federal affairs coverage:** % of labeled federal affairs getting ≥1 primary APS code (target: 100%)
- **PolitMonitor sanity:** share of digitalisation-related APS codes on digi-tagged affairs (lower bound — broad source labels limit precision)

**Known limitations:**
- Coarse Curia Vista labels (28 topics) cannot distinguish narrow APS topics — the mapping captures the family, not the specific topic.
- ZH free keywords are abbreviations (KVG, PBG, ...) that Apertus sometimes misinterprets without affair context.
- 5 APS codes remain unreachable: 2.1.4 Landsgemeinde, 2.1.9 Politikfinanzierung, 4.1.10 Fahrende, 5.2.11 Ergänzungsleistungen, 7.2.10 Versorgungssicherheit — these have no natural source-label counterpart.